In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [2]:
import torch
import triton
import torch.nn.functional as F

In [3]:
from kernels.awa.triton.fwd_kernel_tf32 import awa_tiled_kernel

In [16]:
def run_awa_tf32(q, k, v, meta_tokens, window_size):
    if q.shape[1] > q.shape[2]:
         q = q.transpose(1, 2).contiguous()
         k = k.transpose(1, 2).contiguous()
         v = v.transpose(1, 2).contiguous()
         transposed = True
    else:
         q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
         transposed = False

    # Force inputs to Float32 for TF32 usage
    if q.dtype != torch.float32:
        q = q.float()
        k = k.float()
        v = v.float()
        meta_tokens = meta_tokens.float()

    B, H, L, D = q.shape
    M_val = meta_tokens.shape[0]
    out = torch.empty_like(q)
    stride_m, stride_d = meta_tokens.stride()

    BLOCK_M = 32
    BLOCK_N = 32
    BLOCK_D = triton.next_power_of_2(D)
    next_pow2_M = triton.next_power_of_2(M_val)
    BLOCK_M_META = max(16, next_pow2_M)

    grid = (triton.cdiv(L, BLOCK_M), B * H)

    awa_tiled_kernel[grid](
        q, k, v, meta_tokens, out,
        *q.stride(), *k.stride(), *v.stride(),
        0, 0, stride_m, stride_d,
        *out.stride(),
        B, H, L, D, window_size,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        BLOCK_M_META=BLOCK_M_META, M=M_val,
        num_stages=3, num_warps=4
    )

    return out.transpose(1, 2) if transposed else out

In [17]:
def pytorch_baseline(q, k, v, meta_tokens, window_size):
    # Standard PyTorch implementation for verification
    # q, k, v input shape (B, L, H, D) expected here
    B, L, H, D = q.shape
    scale = 1.0 / (D ** 0.5)

    q = q.transpose(1, 2) # (B, H, L, D)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)

    # Meta Scores
    meta_scores = torch.matmul(q, meta_tokens.t()) * scale

    # Window Scores
    pad_left = window_size - 1
    k_padded = F.pad(k, (0, 0, pad_left, 0))
    v_padded = F.pad(v, (0, 0, pad_left, 0))
    k_win = k_padded.unfold(2, window_size, 1)
    v_win = v_padded.unfold(2, window_size, 1)

    local_scores = torch.matmul(q.unsqueeze(-2), k_win).squeeze(-2) * scale

    mask = torch.ones(L, device=q.device)
    mask_padded = F.pad(mask, (pad_left, 0), value=0)
    mask_win = mask_padded.unfold(0, window_size, 1)
    local_scores = local_scores.masked_fill(mask_win.unsqueeze(0).unsqueeze(0) == 0, float("-inf"))

    max_meta = meta_scores.max(dim=-1, keepdim=True)[0]
    max_local = local_scores.max(dim=-1, keepdim=True)[0]
    max_val = torch.maximum(max_meta, max_local)

    exp_meta = torch.exp(meta_scores - max_val)
    exp_local = torch.exp(local_scores - max_val)

    denom = exp_meta.sum(dim=-1, keepdim=True) + exp_local.sum(dim=-1, keepdim=True)
    numerator = torch.matmul(exp_local.unsqueeze(-2), v_win.transpose(-1, -2)).squeeze(-2)

    out = numerator / (denom + 1e-8)
    return out.transpose(1, 2)

In [18]:

B, L, H, D = 2, 1024, 4, 128
WINDOW_SIZE = 128
NUM_META = 6
dtype = torch.float32
device = "cuda"

print(f"Benchmarking AWA (TF32 on L4): Batch={B}, Len={L}, Heads={H}, Dim={D}")

torch.manual_seed(0)
q = torch.randn((B, L, H, D), device=device, dtype=dtype)
k = torch.randn((B, L, H, D), device=device, dtype=dtype)
v = torch.randn((B, L, H, D), device=device, dtype=dtype)
meta = torch.randn((NUM_META, D), device=device, dtype=dtype)

print("\nVerifying Correctness (Ref FP32 vs Triton TF32)...")

# PyTorch Baseline
ref_out = pytorch_baseline(q, k, v, meta, WINDOW_SIZE)

# Triton TF32
# Ensure allow_tf32 is strictly enabled for PyTorch comparisons if mixing ops,
# but Triton handles it internally via the kernel.
torch.backends.cuda.matmul.allow_tf32 = True
tri_out = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)

diff = (ref_out - tri_out).abs()
max_diff = diff.max().item()
mean_val = ref_out.abs().mean().item()

print(f"   Output Mean Value: {mean_val:.6f}")
print(f"   Max Difference:    {max_diff:.6f}")

# TF32 precision is better than FP16 but worse than strict FP32
# Expect diff around 1e-4 or 1e-5
if max_diff < 5e-3:
    print("PASS: High Precision verified.")
else:
    print(f"Diff is {100 * max_diff / mean_val:.3f}% of mean value.")

print("\nStarting Performance Benchmark...")

# Warmup
for _ in range(10):
    _ = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()
n_loops = 100
for _ in range(n_loops):
    _ = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)
end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops
print(f"Average time per run: {avg_time:.4f} ms")

Benchmarking AWA (TF32 on L4): Batch=2, Len=1024, Heads=4, Dim=128

Verifying Correctness (Ref FP32 vs Triton TF32)...
   Output Mean Value: 0.113018
   Max Difference:    0.003064
PASS: High Precision verified.

Starting Performance Benchmark...
Average time per run: 0.1304 ms
